# MVPA Distinctiveness Analysis
Following Liu et al. 2025 — category-selective representational similarity in VOTC.

**Pipeline:**
1. Define functional ROIs: threshold zstat (differential contrasts) within anatomical search masks, take largest cluster
2. Create 7mm sphere around cluster centroid
3. Extract raw beta patterns (COPEs 15-19) within sphere
4. Compute correlation matrix, Fisher-transform
5. Distinctiveness = mean Fisher(r) between preferred and non-preferred categories

**Lower distinctiveness = more differentiable = better selectivity**

**Notes:**
- All registered data (zstats, copes) are in ses-01 anatomical space. Brain masks always come from ses-01.
- For longitudinal representational change (Cell 8), a FIXED sphere from ses-01 centroid is used to isolate representational change from spatial drift.
- Cross-sectional controls (1 session) contribute to group distinctiveness (Cell 7) but not to drift or representational change (Cells 8-9).
- ROI definition uses differential contrasts approximating Liu (Face>Object, House>Object, Object>Scramble, Word>Scramble).

In [1]:
# CELL 1: Setup & Configuration
import os, sys
import numpy as np
import nibabel as nib
import pandas as pd
import pickle
from pathlib import Path
from scipy.ndimage import label, center_of_mass
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import (processed_dir, skip_subs, is_patient,
                           get_sessions, get_sub_info, _load_csv)

BASE_DIR = Path(processed_dir)
OUTPUT_DIR = BASE_DIR / 'analyses' / 'mvpa_distinctiveness'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ROI definition contrasts (differential, approximating Liu 2025)
# Liu: face>house, house>face, object>scramble, word>face
# Our closest available COPEs:
ROI_COPE_MAP = {
    'face': 1,     # Face > Object
    'house': 2,    # House > Object
    'object': 3,   # Object > Scramble
    'word': 12     # Word > Scramble
}

# Raw beta COPEs for RSA pattern extraction
RSA_COPE_MAP = {
    'face': 15,
    'house': 16,
    'object': 17,
    'word': 18
}

# Parameters
Z_THRESHOLD = 2.3
MIN_CLUSTER_VOXELS = 50
SPHERE_RADIUS_MM = 6  # Liu uses 7mm

BILATERAL_CATEGORIES = ['object', 'house']
UNILATERAL_CATEGORIES = ['face', 'word']

print(f'Base directory: {BASE_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'ROI contrasts: {ROI_COPE_MAP}')
print(f'RSA betas: {RSA_COPE_MAP}')
print(f'Threshold: z>{Z_THRESHOLD}, min {MIN_CLUSTER_VOXELS} voxels')
print(f'Sphere: {SPHERE_RADIUS_MM}mm')

Base directory: /user_data/csimmon2/sym_pt
Output: /user_data/csimmon2/sym_pt/analyses/mvpa_distinctiveness
ROI contrasts: {'face': 1, 'house': 2, 'object': 3, 'word': 12}
RSA betas: {'face': 15, 'house': 16, 'object': 17, 'word': 18}
Threshold: z>2.3, min 50 voxels
Sphere: 6mm


In [ ]:
# CELL 2: Load Subjects from Unified CSV
df = _load_csv()

SUBJECTS = {}
for sub_clean in sorted(df['sub_clean'].unique()):
    if sub_clean in skip_subs:
        continue
    sessions = get_sessions(sub_clean)
    if not sessions:
        continue
    info = get_sub_info(sub_clean, sessions[0])
    pt = is_patient(sub_clean)
    intact = info.get('intact_hemi', '')

    if not pt:
        hemis = ['l', 'r']
    elif intact == 'left':
        hemis = ['l']
    elif intact == 'right':
        hemis = ['r']
    else:
        hemis = ['l', 'r']

    SUBJECTS[sub_clean] = {
        'sessions': sessions,
        'hemispheres': hemis,
        'group': info.get('group', 'unknown'),
        'status': 'patient' if pt else 'control',
        'intact_hemi': intact,
        'surgery_side': info.get('surgery_side', ''),
        'code': info.get('code', '')
    }

n_pt = sum(1 for v in SUBJECTS.values() if v['status'] == 'patient')
n_ctrl = sum(1 for v in SUBJECTS.values() if v['status'] == 'control')
print(f'Loaded {len(SUBJECTS)} subjects: {n_pt} patients, {n_ctrl} controls')

for sub, info in list(SUBJECTS.items())[:5]:
    print(f"  sub-{sub}: {info['group']} {info['status']}, "
          f"hemis={info['hemispheres']}, sessions={info['sessions']}")

In [ ]:
# CELL 3: Helper Functions

def create_sphere(peak_coord, affine, brain_shape, radius=SPHERE_RADIUS_MM):
    """Create spherical ROI in voxel space around a world-coordinate peak."""
    coords = np.array(np.meshgrid(
        np.arange(brain_shape[0]),
        np.arange(brain_shape[1]),
        np.arange(brain_shape[2]),
        indexing='ij'
    )).reshape(3, -1).T

    world = nib.affines.apply_affine(affine, coords)
    dist = np.linalg.norm(world - peak_coord, axis=1)

    mask = np.zeros(brain_shape, dtype=bool)
    within = coords[dist <= radius]
    for c in within:
        mask[c[0], c[1], c[2]] = True
    return mask


def load_brain_mask(sub_clean, ses):
    """Load brain mask for a subject-session."""
    f = BASE_DIR / f'sub-{sub_clean}' / f'ses-{ses:02d}' / 'anat' / 'T1w_brain_mask.nii.gz'
    if f.exists():
        return nib.load(str(f)).get_fdata() > 0
    return None


def get_highlevel_stat(sub_clean, ses, cope_num, first_ses, stat='zstat1'):
    """Get path to HighLevel stat file (zstat or cope)."""
    feat_dir = (BASE_DIR / f'sub-{sub_clean}' / f'ses-{ses:02d}' /
                'derivatives' / 'fsl' / 'loc' / 'HighLevel.gfeat')
    if ses == first_ses:
        return feat_dir / f'cope{cope_num}.feat' / 'stats' / f'{stat}.nii.gz'
    else:
        return feat_dir / f'cope{cope_num}.feat' / 'stats' / f'{stat}_ses{first_ses:02d}.nii.gz'


def find_searchmask(sub_clean, first_ses, hemi, category):
    """Find searchmask — check both ROIs/ and derivatives/rois/."""
    ses_str = f'{first_ses:02d}'
    for subdir in ['ROIs', os.path.join('derivatives', 'rois')]:
        p = BASE_DIR / f'sub-{sub_clean}' / f'ses-{ses_str}' / subdir / f'{hemi}_{category}_searchmask.nii.gz'
        if p.exists():
            return p
    return None


print('Helper functions defined.')

In [ ]:
# CELL 4: Extract Functional ROIs
# NOTE: Brain mask always from first session since all data is registered to ses-01 space

def extract_functional_rois(sub_clean):
    """Find functional ROI clusters within anatomical search masks."""
    info = SUBJECTS[sub_clean]
    sessions = info['sessions']
    hemis = info['hemispheres']
    first_ses = sessions[0]
    results = {}

    # Brain mask from FIRST session only (all data in ses-01 space)
    brain_mask = load_brain_mask(sub_clean, first_ses)

    for hemi in hemis:
        for category, cope_num in ROI_COPE_MAP.items():
            key = f'{hemi}_{category}'
            results[key] = {}

            mask_file = find_searchmask(sub_clean, first_ses, hemi, category)
            if mask_file is None:
                continue

            mask_img = nib.load(str(mask_file))
            mask = mask_img.get_fdata() > 0
            affine = mask_img.affine

            for ses in sessions:
                zstat_file = get_highlevel_stat(sub_clean, ses, cope_num, first_ses, 'zstat1')
                if not zstat_file.exists():
                    continue

                zstat = nib.load(str(zstat_file)).get_fdata()

                # Apply brain mask from first session
                if brain_mask is not None:
                    zstat[~brain_mask] = 0

                suprathresh = (zstat > Z_THRESHOLD) & mask
                if suprathresh.sum() < MIN_CLUSTER_VOXELS:
                    continue

                labeled, n_clusters = label(suprathresh)
                if n_clusters == 0:
                    continue

                sizes = [(labeled == i).sum() for i in range(1, n_clusters + 1)]
                largest = np.argmax(sizes) + 1
                roi_mask = (labeled == largest)

                peak_idx = np.unravel_index(np.argmax(zstat * roi_mask), zstat.shape)
                centroid = nib.affines.apply_affine(affine, center_of_mass(roi_mask))

                results[key][ses] = {
                    'n_voxels': sizes[largest - 1],
                    'peak_z': float(zstat[peak_idx]),
                    'centroid': centroid,
                    'roi_mask': roi_mask,
                    'affine': affine,
                    'brain_shape': mask_img.shape
                }

    return results


print('EXTRACTING FUNCTIONAL ROIs')
print('=' * 70)

all_rois = {}
for sub_clean, info in SUBJECTS.items():
    print(f"\nsub-{sub_clean} ({info['group']} {info['status']}, hemis={info['hemispheres']})")
    try:
        rois = extract_functional_rois(sub_clean)
        all_rois[sub_clean] = rois
        for key, ses_data in rois.items():
            if ses_data:
                sess = sorted(ses_data.keys())
                vox = [ses_data[s]['n_voxels'] for s in sess]
                print(f"  {key}: {len(sess)} sessions, voxels={vox}")
    except Exception as e:
        print(f"  ERROR: {e}")
        all_rois[sub_clean] = {}

n_with_data = sum(1 for v in all_rois.values() if any(sv for sv in v.values()))
print(f'\nExtracted ROIs for {n_with_data}/{len(SUBJECTS)} subjects')

In [ ]:
# CELL 5: RSA — Extract Betas, Compute RDMs, Distinctiveness
# Per-session spheres: each session uses its OWN centroid for cross-sectional distinctiveness

def extract_rsa_betas(sub_clean, ses, sphere_mask, first_ses):
    """Extract raw beta patterns from sphere for all categories."""
    patterns = {}
    for category, cope_num in RSA_COPE_MAP.items():
        cope_file = get_highlevel_stat(sub_clean, ses, cope_num, first_ses, 'cope1')
        if not cope_file.exists():
            continue
        data = nib.load(str(cope_file)).get_fdata()
        betas = data[sphere_mask]
        betas = betas[np.isfinite(betas)]
        if len(betas) > 0:
            patterns[category] = betas
    return patterns


def compute_rdm(patterns):
    """Compute RDM and Fisher-transformed correlation matrix."""
    cats = sorted(patterns.keys())
    if len(cats) < 4:
        return None, None, None
    min_vox = min(len(patterns[c]) for c in cats)
    mat = np.column_stack([patterns[c][:min_vox] for c in cats])
    corr = np.corrcoef(mat.T)
    fisher = np.arctanh(np.clip(corr, -0.999, 0.999))
    return 1 - corr, fisher, cats


def compute_distinctiveness(fisher, cats, pref_category):
    """Mean Fisher(r) between preferred and non-preferred. Lower = more distinct."""
    if fisher is None or pref_category not in cats:
        return None
    idx = cats.index(pref_category)
    nonpref = [i for i, c in enumerate(cats) if c != pref_category]
    return float(np.mean(fisher[idx, nonpref]))


print('COMPUTING RSA DISTINCTIVENESS (per-session spheres)')
print('=' * 70)

all_rsa = {}

for sub_clean, roi_data in all_rois.items():
    info = SUBJECTS[sub_clean]
    first_ses = info['sessions'][0]
    all_rsa[sub_clean] = {}

    for hemi_cat, ses_data in roi_data.items():
        hemi, category = hemi_cat.split('_', 1)
        all_rsa[sub_clean][hemi_cat] = {}

        for ses, rd in ses_data.items():
            # Sphere around THIS session's centroid
            sphere = create_sphere(rd['centroid'], rd['affine'],
                                   rd['brain_shape'], SPHERE_RADIUS_MM)
            patterns = extract_rsa_betas(sub_clean, ses, sphere, first_ses)
            rdm, fisher, cats = compute_rdm(patterns)
            distinct = compute_distinctiveness(fisher, cats, category) if fisher is not None else None

            all_rsa[sub_clean][hemi_cat][ses] = {
                'centroid': rd['centroid'],
                'n_voxels': rd['n_voxels'],
                'peak_z': rd['peak_z'],
                'sphere_voxels': int(sphere.sum()),
                'rdm': rdm,
                'fisher': fisher,
                'categories': cats,
                'distinctiveness': distinct
            }

            tag = f'd={distinct:.3f}' if distinct is not None else 'no RSA'
            print(f"  sub-{sub_clean} {hemi_cat} ses-{ses:02d}: "
                  f"{rd['n_voxels']} vox, z={rd['peak_z']:.1f}, {tag}")

print('\nRSA complete.')

In [ ]:
# CELL 6: Compile Results Tables

def compile_results_table():
    """Compile all results into a DataFrame."""
    rows = []
    for sub_clean, hemi_data in all_rsa.items():
        info = SUBJECTS[sub_clean]
        for hemi_cat, ses_data in hemi_data.items():
            hemi, category = hemi_cat.split('_', 1)
            hemi_name = 'left' if hemi == 'l' else 'right'
            cat_type = 'Bilateral' if category in BILATERAL_CATEGORIES else 'Unilateral'

            for ses, m in ses_data.items():
                rows.append({
                    'Subject': f'sub-{sub_clean}',
                    'Group': info['group'],
                    'Status': info['status'],
                    'Hemisphere': hemi_name,
                    'Category': category,
                    'Category_Type': cat_type,
                    'Session': ses,
                    'N_Voxels': m['n_voxels'],
                    'Peak_Z': m['peak_z'],
                    'Sphere_Voxels': m['sphere_voxels'],
                    'Distinctiveness': m['distinctiveness'],
                    'Centroid_X': m['centroid'][0],
                    'Centroid_Y': m['centroid'][1],
                    'Centroid_Z': m['centroid'][2],
                })
    return pd.DataFrame(rows)


results_df = compile_results_table()

print(f'Results table: {len(results_df)} rows')
print(f'Subjects with distinctiveness: '
      f'{results_df.dropna(subset=["Distinctiveness"])["Subject"].nunique()}')
results_df.head(10)

In [ ]:
# CELL 7: Group Analysis — Cross-Sectional Distinctiveness

def group_summary(results_df):
    """Three-group comparison: OTC vs nonOTC vs Controls."""
    valid = results_df.dropna(subset=['Distinctiveness'])

    # Controls: average across hemispheres per subject/category/session
    ctrl = valid[valid['Status'] == 'control'].copy()
    ctrl_avg = ctrl.groupby(['Subject', 'Category', 'Category_Type', 'Session']).agg({
        'Distinctiveness': 'mean'
    }).reset_index()

    patients = valid[valid['Status'] == 'patient']
    otc = patients[patients['Group'] == 'OTC']
    nonotc = patients[patients['Group'] == 'nonOTC']

    print('THREE-GROUP DISTINCTIVENESS COMPARISON')
    print('=' * 60)
    print(f"{'Group':<15} {'Bilateral':<12} {'Unilateral':<12} {'Difference':<12}")
    print('-' * 52)

    for name, data in [('OTC', otc), ('nonOTC', nonotc), ('Controls', ctrl_avg)]:
        if len(data) == 0:
            print(f'{name:<15} no data')
            continue
        bil = data[data['Category_Type'] == 'Bilateral']['Distinctiveness'].mean()
        uni = data[data['Category_Type'] == 'Unilateral']['Distinctiveness'].mean()
        diff = bil - uni
        print(f'{name:<15} {bil:<12.3f} {uni:<12.3f} {diff:<12.3f}')

    # Controls by hemisphere
    print(f'\n  Controls by hemisphere:')
    ctrl_full = valid[valid['Status'] == 'control']
    for h in ['left', 'right']:
        hd = ctrl_full[ctrl_full['Hemisphere'] == h]
        if len(hd) == 0:
            continue
        bil = hd[hd['Category_Type'] == 'Bilateral']['Distinctiveness'].mean()
        uni = hd[hd['Category_Type'] == 'Unilateral']['Distinctiveness'].mean()
        print(f"    {h:<8} {bil:<12.3f} {uni:<12.3f} {bil-uni:<12.3f}")


group_summary(results_df)

In [ ]:
# CELL 8: Representational Change Over Time (Longitudinal)
# IMPORTANT: Uses FIXED sphere from session 1 centroid to isolate
# representational change from spatial drift.

def compute_representational_change_fixed_sphere():
    """Re-extract RSA with fixed ses-01 sphere for longitudinal subjects."""
    rows = []

    for sub_clean, roi_data in all_rois.items():
        info = SUBJECTS[sub_clean]
        first_ses = info['sessions'][0]

        for hemi_cat, ses_data in roi_data.items():
            hemi, category = hemi_cat.split('_', 1)
            sess = sorted(ses_data.keys())

            if len(sess) < 2:
                continue

            # Anchor sphere to FIRST session centroid
            if first_ses not in ses_data:
                continue

            anchor = ses_data[first_ses]
            fixed_sphere = create_sphere(
                anchor['centroid'], anchor['affine'],
                anchor['brain_shape'], SPHERE_RADIUS_MM
            )

            # Extract distinctiveness at each session using fixed sphere
            session_distinct = {}
            for ses in sess:
                patterns = extract_rsa_betas(sub_clean, ses, fixed_sphere, first_ses)
                _, fisher, cats = compute_rdm(patterns)
                d = compute_distinctiveness(fisher, cats, category)
                if d is not None:
                    session_distinct[ses] = d

            if len(session_distinct) < 2:
                continue

            sd_sessions = sorted(session_distinct.keys())
            first_d = session_distinct[sd_sessions[0]]
            last_d = session_distinct[sd_sessions[-1]]
            change = abs(last_d - first_d)

            rows.append({
                'Subject': f'sub-{sub_clean}',
                'Group': info['group'],
                'Status': info['status'],
                'Hemisphere': 'left' if hemi == 'l' else 'right',
                'Category': category,
                'Category_Type': 'Bilateral' if category in BILATERAL_CATEGORIES else 'Unilateral',
                'First_Session': sd_sessions[0],
                'Last_Session': sd_sessions[-1],
                'First_Distinctiveness': first_d,
                'Last_Distinctiveness': last_d,
                'Representational_Change': change,
                'N_Sessions': len(sd_sessions)
            })

    return pd.DataFrame(rows)


repr_change_df = compute_representational_change_fixed_sphere()

if len(repr_change_df) > 0:
    print('REPRESENTATIONAL CHANGE (fixed ses-01 sphere)')
    print('=' * 60)
    print(f"{'Group':<15} {'Bilateral':<12} {'Unilateral':<12} {'Difference':<12}")
    print('-' * 52)

    ctrl_rc = repr_change_df[repr_change_df['Status'] == 'control'].copy()
    ctrl_avg = ctrl_rc.groupby(['Subject', 'Category', 'Category_Type']).agg({
        'Representational_Change': 'mean'
    }).reset_index() if len(ctrl_rc) > 0 else pd.DataFrame()

    pt_rc = repr_change_df[repr_change_df['Status'] == 'patient']

    for name, data in [('OTC', pt_rc[pt_rc['Group'] == 'OTC']),
                       ('nonOTC', pt_rc[pt_rc['Group'] == 'nonOTC']),
                       ('Controls', ctrl_avg)]:
        if len(data) == 0:
            print(f'{name:<15} no longitudinal data')
            continue
        bil = data[data['Category_Type'] == 'Bilateral']['Representational_Change'].mean()
        uni = data[data['Category_Type'] == 'Unilateral']['Representational_Change'].mean()
        diff = bil - uni
        print(f'{name:<15} {bil:<12.3f} {uni:<12.3f} {diff:<12.3f}')
else:
    print('No longitudinal data available yet.')

In [ ]:
# CELL 9: Spatial Drift Summary

def compile_drift_table():
    """Compute spatial drift from baseline session."""
    rows = []
    for sub_clean, hemi_data in all_rsa.items():
        info = SUBJECTS[sub_clean]
        for hemi_cat, ses_data in hemi_data.items():
            hemi, category = hemi_cat.split('_', 1)
            sess = sorted(ses_data.keys())
            if len(sess) < 2:
                continue
            baseline = ses_data[sess[0]]['centroid']
            for ses in sess[1:]:
                current = ses_data[ses]['centroid']
                drift = float(np.linalg.norm(np.array(current) - np.array(baseline)))
                rows.append({
                    'Subject': f'sub-{sub_clean}',
                    'Group': info['group'],
                    'Status': info['status'],
                    'Hemisphere': 'left' if hemi == 'l' else 'right',
                    'Category': category,
                    'Category_Type': 'Bilateral' if category in BILATERAL_CATEGORIES else 'Unilateral',
                    'Baseline_Session': sess[0],
                    'Session': ses,
                    'Drift_mm': drift
                })
    return pd.DataFrame(rows)


drift_df = compile_drift_table()

if len(drift_df) > 0:
    print('SPATIAL DRIFT SUMMARY')
    print('=' * 60)

    for group in ['OTC', 'nonOTC', 'control']:
        gd = drift_df[drift_df['Status'] == 'control'] if group == 'control' else drift_df[drift_df['Group'] == group]
        if len(gd) == 0:
            continue
        print(f'\n{group.upper()}:')
        for ct in ['Bilateral', 'Unilateral']:
            ctd = gd[gd['Category_Type'] == ct]
            if len(ctd) > 0:
                print(f'  {ct}: mean={ctd["Drift_mm"].mean():.1f}mm, '
                      f'sd={ctd["Drift_mm"].std():.1f}mm, n={len(ctd)}')
else:
    print('No drift data available yet.')

In [ ]:
# CELL 10: Verification

print('VERIFICATION')
print('=' * 60)

print(f'\n1. BASIC COUNTS:')
print(f'   Total result rows: {len(results_df)}')
print(f'   Unique subjects: {results_df["Subject"].nunique()}')
n_distinct = results_df.dropna(subset=['Distinctiveness'])['Subject'].nunique()
print(f'   With distinctiveness: {n_distinct}')

print(f'\n2. GROUP BREAKDOWN:')
for group in sorted(results_df['Group'].unique()):
    gd = results_df[results_df['Group'] == group]
    print(f'   {group}: {gd["Subject"].nunique()} subjects, {len(gd)} rows')

print(f'\n3. HEMISPHERE COVERAGE:')
for status in ['patient', 'control']:
    sd = results_df[results_df['Status'] == status]
    for h in ['left', 'right']:
        hd = sd[sd['Hemisphere'] == h]
        print(f'   {status} {h}: {hd["Subject"].nunique()} subjects')

print(f'\n4. CATEGORY COVERAGE:')
for cat in ['face', 'word', 'object', 'house']:
    cd = results_df[results_df['Category'] == cat]
    n_d = cd.dropna(subset=['Distinctiveness']).shape[0]
    print(f'   {cat}: {len(cd)} entries, {n_d} with distinctiveness')

print(f'\n5. LONGITUDINAL SUBJECTS:')
for subj in sorted(results_df['Subject'].unique()):
    sd = results_df[results_df['Subject'] == subj]
    n_ses = sd['Session'].nunique()
    if n_ses > 1:
        print(f'   {subj}: {n_ses} sessions')

print(f'\n6. ANALYSIS PARAMETERS:')
print(f'   ROI contrasts: {ROI_COPE_MAP}')
print(f'   RSA betas: {RSA_COPE_MAP}')
print(f'   Z threshold: {Z_THRESHOLD}')
print(f'   Sphere radius: {SPHERE_RADIUS_MM}mm')
print(f'   Min cluster: {MIN_CLUSTER_VOXELS} voxels')

In [ ]:
# CELL 11: Save Results (to cluster, NOT git)

results_df.to_csv(OUTPUT_DIR / 'rsa_results.csv', index=False)

if len(drift_df) > 0:
    drift_df.to_csv(OUTPUT_DIR / 'spatial_drift.csv', index=False)

if len(repr_change_df) > 0:
    repr_change_df.to_csv(OUTPUT_DIR / 'representational_change.csv', index=False)

# Pickle: strip large arrays to save space
save_data = {
    'all_rois': {s: {k: {ses: {kk: vv for kk, vv in v.items() if kk != 'roi_mask'}
                         for ses, v in sv.items()}
                     for k, sv in sd.items()}
                for s, sd in all_rois.items()},
    'all_rsa': all_rsa,
    'results_df': results_df,
    'drift_df': drift_df,
    'repr_change_df': repr_change_df,
    'config': {
        'ROI_COPE_MAP': ROI_COPE_MAP,
        'RSA_COPE_MAP': RSA_COPE_MAP,
        'Z_THRESHOLD': Z_THRESHOLD,
        'SPHERE_RADIUS_MM': SPHERE_RADIUS_MM,
        'MIN_CLUSTER_VOXELS': MIN_CLUSTER_VOXELS
    }
}

with open(OUTPUT_DIR / 'mvpa_distinctiveness_results.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print(f'Saved to: {OUTPUT_DIR}')
print(f'  rsa_results.csv ({len(results_df)} rows)')
if len(drift_df) > 0:
    print(f'  spatial_drift.csv ({len(drift_df)} rows)')
if len(repr_change_df) > 0:
    print(f'  representational_change.csv ({len(repr_change_df)} rows)')
print(f'  mvpa_distinctiveness_results.pkl')
print(f'\nDone!')